#### Assignment #3
#### Alia Kasem 
`All Foundation of empirical research`

1. loading the needed data: diamonds, flight, airports 
2. Filtering task 
3. Aggregations 
4. Join exercises 
5. Visualizations 


In [0]:
# Diamonds dataset
diamonds_df = spark.read.csv(
    "/Volumes/databricks_rdatasets_compilation/vf58caff/csv/ggplot2/diamonds.csv",
    header=True,
    inferSchema=True
)
for col_name in diamonds_df.columns:
    diamonds_df = diamonds_df.withColumnRenamed(col_name, col_name.replace(".", "_"))
diamonds_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("default.diamonds")

# Flights dataset
flights_df = spark.read.csv(
    "/Volumes/databricks_rdatasets_compilation/vf58caff/csv/nycflights13/flights.csv",
    header=True,
    inferSchema=True
)
for col_name in flights_df.columns:
    flights_df = flights_df.withColumnRenamed(col_name, col_name.replace(".", "_"))
flights_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("default.flights")

# Airports dataset
airports_df = spark.read.csv(
    "/Volumes/databricks_rdatasets_compilation/vf58caff/csv/nycflights13/airports.csv",
    header=True,
    inferSchema=True
)
for col_name in airports_df.columns:
    airports_df = airports_df.withColumnRenamed(col_name, col_name.replace(".", "_"))
airports_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("default.airports")

# PySpark DataFrames for visualization
diamonds = spark.table("default.diamonds")
flights = spark.table("default.flights")
airports = spark.table("default.airports")

In [0]:

from pyspark.sql.functions import col
# 1a. Diamonds: Ideal cut AND over 2 carats
ideal_diamonds_2carat = diamonds.filter((col("cut") == "Ideal") & (col("carat") > 2))
display(ideal_diamonds_2carat)
# 1b. Flights: Departed in December with dep_delay > 2 hours
december_delays = flights.filter((col("month") == 12) & (col("dep_delay") > 120))
display(december_delays)

In [0]:

from pyspark.sql.functions import avg, round
# Average diamond price grouped by cut AND color
avg_price_df = diamonds.groupBy("cut", "color") \
    .agg(round(avg("price"),2).alias("avg_price")) \
    .orderBy("cut", "color")
display(avg_price_df) 


In [0]:
# Month with worst average arrival delays
avg_delay_month = flights.groupBy("month") \
    .agg(round(avg("arr_delay"),2).alias("avg_arr_delay")) \
    .orderBy(col("avg_arr_delay").desc())
display(avg_delay_month)  # Bar chart 

Databricks visualization. Run in Databricks to view.

In [0]:
# Top 10 destination airports with worst average arrival delays
worst_airports = flights.join(airports, flights.dest == airports.faa) \
    .groupBy("name") \
    .agg(round(avg("arr_delay"),2).alias("avg_arr_delay")) \
    .orderBy(col("avg_arr_delay").desc()) \
    .limit(10)
display(worst_airports)  # Horizontal bar chart

In [0]:
# Scatter plot: Price vs Carat for Ideal diamonds > 2 carats
display(ideal_diamonds_2carat.select("carat", "price", "color"))  # Scatter plot, color by 'color'

# Boxplot: Arrival delay distribution per month
display(flights.select("month", "arr_delay"))  # Boxplot

# Bar chart: Average arrival delay by month
display(avg_delay_month)  # Bar chart

# Horizontal bar chart: Top 10 airports with worst delays
display(worst_airports)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

`Key Findings`
- `Worst Avg of arrival delay: Columbia Metropolitan Airport (CAE) Avg, ~41.74 min`

- `After joining flights with airports and calculating average arrival delays, I found that Columbia Metropolitan (CAE) had the worst average arrival delay at 41.76 minutes. Several mid-sized regional airports like Tulsa (TUL) and Oklahoma City (OKC) also showed significantly high delays. Interestingly, major hub airports did not rank among the worst average delays.`

SQL 

In [0]:
# Load flights table into a PySpark DataFrame
flights = spark.table("default.flights")

# Load diamonds if you want to visualize them too
diamonds = spark.table("default.diamonds")


In [0]:
from pyspark.sql.functions import round, avg, col

avg_delay_month = flights.groupBy("month") \
    .agg(round(avg("arr_delay"), 2).alias("avg_arr_delay")) \
    .orderBy("month")

display(avg_delay_month) 

In [0]:
from pyspark.sql.functions import round, avg, col

avg_delay_month = flights.groupBy("month") \
    .agg(round(avg("arr_delay"), 2).alias("avg_arr_delay")) \
    .orderBy("month")

display(avg_delay_month)  

In [0]:
from pyspark.sql.functions import round, avg

avg_price = diamonds.groupBy("cut", "color") \
    .agg(round(avg("price"),2).alias("avg_price"))

display(avg_price)  # Heatmap option in Databricks